In [1]:
import os
import json

from jinja2.compiler import generate

In [2]:
sentinel_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\sentinel 2\sentinel 2\sentinel2_aois_all_bands_OrigRes".replace("\\","/")

landsat_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\landsat\bands_from_B2_to_B6".replace("\\","/")

lis3_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\bhoonidhi\Lis3\croped_bands_tifs".replace("\\","/")

lis4_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\bhoonidhi\lis4\all_cropped_bands".replace("\\","/")

In [5]:
import os, json

class ans_gen:
    def __init__(
            self,

    ):

        self.sentinel_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\sentinel 2\sentinel 2\sentinel2_aois_all_bands_OrigRes".replace("\\","/")

        self.landsat_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\landsat\bands_from_B2_to_B6".replace("\\","/")

        self.lis3_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\bhoonidhi\Lis3\croped_bands_tifs".replace("\\","/")

        self.lis4_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\bhoonidhi\lis4\all_cropped_bands".replace("\\","/")

        self.question_dict_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\questions for testing VLM\questions_version_2.json".replace("\\","/")


        self.questions_dict=None
        with open(self.question_dict_path,"r", encoding="utf-8") as f:
            self.questions_dict=json.load(f)

        self.water_bodies_info=None
        self.veg_stats=None
        self.built_up_stats=None
        self.overall_stats=None
        self.location_info=None
        self.bounds_info=None
        self.old_generated_answers=None



    def open_data_from_dir(
            self,
            dir_path:str
    ):
        water_bodies_info_path=os.path.join(
            dir_path,
            "water_bodies_info.json",
        )
        veg_stats_path=os.path.join(
            dir_path,
            "veg_stats.json",
        )
        built_up_stats_path=os.path.join(
            dir_path,
            "built_up_stats.json",
        )
        overall_land_stats_path=os.path.join(
            dir_path,
            "overall_land_stats.json",
        )
        location_info_path=os.path.join(
            dir_path,
            "location_info.json",
        )
        bounds_path=os.path.join(
            dir_path,
            "bounds.json",
        )
        old_generated_answers_path=os.path.join(
            dir_path,
            "generated_answers.json",
                )

        with open(water_bodies_info_path,"r", encoding="utf-8") as f:
            self.water_bodies_info=json.load(f)

        with open(veg_stats_path,"r", encoding="utf-8") as f:
            self.veg_stats=json.load(f)

        with open(built_up_stats_path,"r", encoding="utf-8") as f:
            self.built_up_stats=json.load(f)

        with open(overall_land_stats_path,"r", encoding="utf-8") as f:
            self.overall_stats=json.load(f)

        with open(location_info_path,"r", encoding="utf-8") as f:
            self.location_info=json.load(f)

        with open(bounds_path,"r", encoding="utf-8") as f:
            self.bounds_info=json.load(f)

        with open(old_generated_answers_path,"r", encoding="utf-8") as f:
                    self.old_generated_answers=json.load(f)



    def answer_gen_object_counting(
            self,
            ques:str
    ):
        if ques=="How many distinct water bodies are present (give only a number)?":
            return len(self.water_bodies_info.keys())
        
        elif ques == "How many distinct land cover classes are present (give only a number)?":
            return len(self.overall_stats["over all land cover in km2"]["info"].keys())
        
        else: raise ValueError(f"{ques} not found in object counting.")
        
    def answer_gen_object_area(
            self,
            ques:str
    ):


        if ques=="What is the total water body area (give one word answer in km2)?":
            return self.overall_stats["over all land cover in km2"]["info"].get("water coverage area in km2",0)

        elif ques=="What percentage of total area is water (give one word answer in %)?":
            return self.overall_stats["over all land cover %"]["info"]["water coverage"]

        

        elif ques == "What is the area of each water body?":
            if len(self.water_bodies_info)==0: return None
            area_in_m2={
                water_body_id: info.get("area in m2",0)
                for water_body_id, info in self.water_bodies_info.items()
            }

            area_in_km2={
                water_body_id: info["area in m2"]/1e6
                for water_body_id, info in self.water_bodies_info.items()
            }

            return (area_in_m2, area_in_km2)

        elif ques == "What percentage of water and non-water area?":
            water_percent=self.overall_stats["over all land cover %"]["info"]["water coverage"]
            other_percent=self.overall_stats["over all land cover %"]["info"]["unclassified land"] + \
                self.overall_stats["over all land cover %"]["info"]["moderate vegetation coverage"] + \
                self.overall_stats["over all land cover %"]["info"]["dense vegetation coverage"] + \
                self.overall_stats["over all land cover %"]["info"].get("moderate built-up coverage",0) + \
                self.overall_stats["over all land cover %"]["info"].get("dense built-up coverage",0)

            return water_percent,other_percent


        elif ques == "What is the total vegetation area (give one word answer in km2)?":
            veg_from_overall = self.overall_stats["over all land cover in km2"]["info"]["moderate vegetation coverage area in km2"] + self.overall_stats["over all land cover in km2"]["info"]["dense vegetation coverage area in km2"]

            # veg_from_veg_stats= [info["vegetation class coverage area in km2"]["coverage_info"][""] for cell,info in self.veg_stats.items()]

            return veg_from_overall

        elif ques == "What percentage of area is vegetated (give one word answer in %)?":
            return self.overall_stats["over all land cover %"]["info"]["moderate vegetation coverage"] + \
                self.overall_stats["over all land cover %"]["info"]["dense vegetation coverage"]

        elif ques == "What percentage of area is vegetated vs. non-vegetated?":
            return self.overall_stats["over all land cover %"]["info"]["water coverage"] +\
                self.overall_stats["over all land cover %"]["info"]["unclassified land"] + \
                self.overall_stats["over all land cover %"]["info"].get("moderate built-up coverage",0) + \
                self.overall_stats["over all land cover %"]["info"].get("dense built-up coverage",0)

        elif ques == "What percentage of area is vegetated vs. water?":
            veg_percent=self.overall_stats["over all land cover %"]["info"]["moderate vegetation coverage"] + \
                self.overall_stats["over all land cover %"]["info"]["dense vegetation coverage"]

            water_percent=self.overall_stats["over all land cover %"]["info"]["water coverage"]

            return veg_percent, water_percent

        elif ques == "What percentage of area is vegetated vs. built-up?":
            veg_percent=self.overall_stats["over all land cover %"]["info"]["moderate vegetation coverage"] + \
                self.overall_stats["over all land cover %"]["info"]["dense vegetation coverage"]

            built_up_percent=self.overall_stats["over all land cover %"]["info"].get("moderate built-up coverage",0) + \
                self.overall_stats["over all land cover %"]["info"].get("dense built-up coverage",0)

            return veg_percent, built_up_percent


        elif ques == "What is the total built-up area (give one word answer in km2)?":
            built_up_from_overall = self.overall_stats["over all land cover in km2"]["info"].get("moderate built-up coverage area in km2",0) + self.overall_stats["over all land cover in km2"]["info"].get("dense built-up coverage area in km2",0)

            return built_up_from_overall

        elif ques == "What percentage of area is built-up (give one word answer in %)?":
            return self.overall_stats["over all land cover %"]["info"].get("moderate built-up coverage",0) + \
                self.overall_stats["over all land cover %"]["info"].get("dense built-up coverage",0)

        elif ques == "What percentage of area is built-up vs. non-built-up?":
            built_up_percent=self.overall_stats["over all land cover %"]["info"].get("moderate built-up coverage",0) + \
                self.overall_stats["over all land cover %"]["info"].get("dense built-up coverage",0)

            non_built_up_percent=self.overall_stats["over all land cover %"]["info"]["water coverage"] +\
                self.overall_stats["over all land cover %"]["info"]["unclassified land"] + \
                self.overall_stats["over all land cover %"]["info"]["moderate vegetation coverage"] + \
                self.overall_stats["over all land cover %"]["info"]["dense vegetation coverage"]

            return built_up_percent, non_built_up_percent

        elif ques == "What percentage of area is built-up vs water?":
            built_up_percent=self.overall_stats["over all land cover %"]["info"].get("moderate built-up coverage",0) + \
                self.overall_stats["over all land cover %"]["info"].get("dense built-up coverage",0)

            water_percent=self.overall_stats["over all land cover %"]["info"]["water coverage"]

            return built_up_percent, water_percent


        elif ques == "What is the proportion of each land cover type?":
            return self.overall_stats["over all land cover %"]["info"]

        elif ques == "What is the area of each land cover type?":
            return self.overall_stats["over all land cover in km2"]["info"]

        else: raise ValueError(f"{ques}not found in object counting and area.")
        

    def answer_gen_object_localization(
            self,
            ques:str
    ):
        cell_name = {
            "cell_1": "upper left", "cell_2": "upper middle", "cell_3": "upper right",
            "cell_4": "middle left", "cell_5": "center", "cell_6": "middle right",
            "cell_7": "lower left", "cell_8": "lower middle", "cell_9": "lower right",
        }

        # if ques == "Where is bare-land located in the image?":
        #     pass

        if ques == "Where is vegetation located in the image?":
            veg_locations=[
                cell_name[cell]
                for cell, info in self.veg_stats.items()

                if info["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation", 0) > 0 or
                   info["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation", 0) > 0
            ]

            return veg_locations

        # elif ques == "Where is vegetation located relative to built-up?":
        #     pass

        # elif ques == "Where is vegetation located relative to water?":
        #     pass

        elif ques == "Where are built-up located?":
            built_up_locations=[
                cell_name[cell]
                for cell, info in self.built_up_stats.items()

                if info.get("built-up class coverage %", False) and (info["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or info["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0)
            ]

            return built_up_locations

        # elif ques == "Where are built-up relative to water bodies?":
        #     pass

        # elif ques == "Where are built-up relative to vegetation?":
        #     pass

        elif ques == "Where are water bodies located in the image?":
            return [info["bounding box"] for info in list(self.water_bodies_info.values())] if len( list(self.water_bodies_info.values()))>0 else None

        # elif ques == "Where are water relative to built-up?":
        #     pass

        # elif ques == "Where are water relative to vegetation?":
        #     pass

        elif ques == "Where is the most vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":

            # cell_total_veg=dict()
            # for cell, info in self.veg_stats.items():
            #     curr_total_veg=0
            #     for veg_type,coverage in info["vegetation class coverage area in km2"]["coverage_info"].items():
            #         if veg_type in ["moderate vegetation", "dense vegetation"]:
            #             curr_total_veg+=coverage
            # 
            #     cell_total_veg[cell]=curr_total_veg


            cell_total_veg={ cell: (info["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) + info["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation", 0)) for cell, info in self.veg_stats.items()}
            if len(cell_total_veg)==0: return None

            max_veg_cell=max(
                cell_total_veg,
                key=lambda k: cell_total_veg[k]
            )

            return cell_name[max_veg_cell]

        elif ques == "Where is the least vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            cell_total_veg={ cell: (info["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation", 0) +
                                    info["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation", 0)) for cell, info in self.veg_stats.items()}

            # cell_total_veg=dict()
            # for cell, info in self.veg_stats.items():
            #     curr_total_veg=0
            #     for veg_type,coverage in info["vegetation class coverage area in km2"]["coverage_info"].items():
            #         if veg_type in ["moderate vegetation", "dense vegetation"]:
            #             curr_total_veg+=coverage
            # 
            #     cell_total_veg[cell]=curr_total_veg

            if len(cell_total_veg)==0: return None

            min_veg_cell=min(
                cell_total_veg,
                key=lambda k: cell_total_veg[k]
            )

            return cell_name[min_veg_cell]

        elif ques == "Where is the most dense vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            cell_total_veg={ cell: (info["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0)) for cell, info in self.veg_stats.items()}

            # cell_total_veg=dict()
            # for cell, info in self.veg_stats.items():
            #     curr_total_veg=0
            #     for veg_type,coverage in info["vegetation class coverage area in km2"]["coverage_info"].items():
            #         if veg_type in ["dense vegetation"]:
            #             curr_total_veg+=coverage

            if len(cell_total_veg)==0: return None

            max_veg_cell=max(
                cell_total_veg,
                key=lambda k: cell_total_veg[k]
            )

            return cell_name[max_veg_cell]

        elif ques == "Where is the most sparse vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            # cell_total_veg={ cell: (info["vegetation class coverage area in km2"]["coverage_info"]["bare ground or sparse vegetation"]) for cell, info in self.veg_stats.items()}
            cell_total_veg=dict()
            for cell, info in self.veg_stats.items():
                curr_total_veg=0
                for veg_type,coverage in info["vegetation class coverage area in km2"]["coverage_info"].items():
                    if veg_type in ["bare ground or sparse vegetation"]:
                        curr_total_veg+=coverage

                cell_total_veg[cell]=curr_total_veg

            if len(cell_total_veg)==0: return None

            max_veg_cell=max(
                cell_total_veg,
                key=lambda k: cell_total_veg[k]
            )

            return cell_name[max_veg_cell]

        elif ques == "Which area has the most fragmented vegetation cover (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            cell_mornas_I={ cell: info["global morans I"][0] for cell, info in self.veg_stats.items() if info["global morans I"][1] < 0.005 }


            min_mornas_cell=min(
                cell_mornas_I,
                key=lambda k: cell_mornas_I[k]
            )

            return cell_name[min_mornas_cell]

        elif ques == "Which area has most continuous vegetation cover (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            cell_mornas_I={ cell: info["global morans I"][0] for cell, info in self.veg_stats.items()  if info["global morans I"][1] < 0.005 }

            max_mornas_cell=max(
                cell_mornas_I,
                key=lambda k: cell_mornas_I[k]
            )

            return cell_name[max_mornas_cell]

        elif ques == "Where is the most built-up in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":

            cell_total_built={ cell: (info["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) + info["built-up class coverage %"]["coverage"].get("High-density built-up areas",0)) * info["total area covered by cell in km2"] / 100 for cell, info in self.built_up_stats.items() if info.get("built-up class coverage %", False)}

            if len(cell_total_built)==0: return None

            max_built_cell=max(
                cell_total_built,
                key=lambda k: cell_total_built[k]
            )

            return cell_name[max_built_cell]

        elif ques == "Where is the least built-up in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            cell_total_built={ cell: (info["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) + info["built-up class coverage %"]["coverage"].get("High-density built-up areas",0)) * info["total area covered by cell in km2"] / 100 for cell, info in self.built_up_stats.items() if info.get("built-up class coverage %", False)}

            if len(cell_total_built)==0: return None

            min_built_cell=min(
                cell_total_built,
                key=lambda k: cell_total_built[k]
            )

            return cell_name[min_built_cell]

        elif ques == "Where is the most dense built-up in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?":
            cell_total_dense_built={ cell: info["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) * info["total area covered by cell in km2"] / 100 for cell, info in self.built_up_stats.items() if info.get("built-up class coverage %", False)}

            if len(cell_total_dense_built)==0: return None

            max_dense_built_cell=max(
                cell_total_dense_built,
                key=lambda k: cell_total_dense_built[k]
            )

            return cell_name[max_dense_built_cell]

        elif ques == "Give location of the largest water body.":

            if len(self.water_bodies_info)==0: return None

            largest_water_body=max(
                self.water_bodies_info,
                key=lambda k: self.water_bodies_info[k]["area in m2"]
            )

            return self.water_bodies_info[largest_water_body]["bounding box"]

        elif ques == "Give location of the smallest water body.":

            if len(self.water_bodies_info)==0: return None

            smallest_water_body=min(
                self.water_bodies_info,
                key=lambda k: self.water_bodies_info[k]["area in m2"]
            )

            return self.water_bodies_info[smallest_water_body]["bounding box"]

        else: raise ValueError(f"{ques} not found in object localization.")


    def answer_gen_object_presence(
            self,
            ques:str
    ):
        if ques == "Is water present in the image (yes/no)?":
            return True if self.overall_stats["over all land cover in km2"]["info"]["water coverage area in km2"] > 0 else False

        elif ques == "Is vegetation present (yes/no)?":
            return True if (self.overall_stats["over all land cover in km2"]["info"]["moderate vegetation coverage area in km2"] > 0 or self.overall_stats["over all land cover in km2"]["info"]["dense vegetation coverage area in km2"] > 0) else False

        elif ques == "Is dense vegetation present (yes/no)?":
            return True if self.overall_stats["over all land cover in km2"]["info"]["dense vegetation coverage area in km2"] > 0 else False

        elif ques == "Is built-up present in the image (yes/no)?":
            return True if (self.overall_stats["over all land cover in km2"]["info"].get("moderate built-up coverage area in km2",0) > 0 or self.overall_stats["over all land cover in km2"]["info"].get("dense built-up coverage area in km2",0) > 0) else False

        elif ques == "Is vegetation present in upper left (yes/no)?":
            return True if (self.veg_stats["cell_1"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_1"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in upper middle (yes/no)?":
            return True if (self.veg_stats["cell_2"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_2"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in upper right (yes/no)?":
            return True if (self.veg_stats["cell_3"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_3"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in middle left (yes/no)?":
            return True if (self.veg_stats["cell_4"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_4"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in center (yes/no)?":
            return True if (self.veg_stats["cell_5"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_5"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in middle right (yes/no)?":
            return True if (self.veg_stats["cell_6"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_6"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in lower left (yes/no)?":
            return True if (self.veg_stats["cell_7"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_7"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in lower middle (yes/no)?":
            return True if (self.veg_stats["cell_8"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_8"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is vegetation present in lower right (yes/no)?":
            return True if (self.veg_stats["cell_9"]["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) > 0 or self.veg_stats["cell_9"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0) else False

        elif ques == "Is dense vegetation present in upper left (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_1"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_1"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in upper middle (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_2"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_2"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in upper right (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_3"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_3"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in middle left (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_4"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_4"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in center (yes/no)?":
            if "dense vegetation" not in list(self.veg_stats["cell_5"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_5"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in middle right (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_6"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_6"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in lower left (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_7"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_7"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in lower middle (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_8"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_8"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is dense vegetation present in lower right (yes/no)?":

            if "dense vegetation" not in list(self.veg_stats["cell_9"]["vegetation class coverage area in km2"]["coverage_info"].keys()):
                return False

            return True if self.veg_stats["cell_9"]["vegetation class coverage area in km2"]["coverage_info"].get("dense vegetation",0) > 0 else False

        elif ques == "Is built-up present in upper left (yes/no)?":
            return True if self.built_up_stats["cell_1"].get("built-up class coverage %",False) and (self.built_up_stats["cell_1"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_1"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in upper middle (yes/no)?":
            return True if self.built_up_stats["cell_2"].get("built-up class coverage %",False) and (self.built_up_stats["cell_2"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_2"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in upper right (yes/no)?":
            return True if self.built_up_stats["cell_3"].get("built-up class coverage %",False) and (self.built_up_stats["cell_3"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_3"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in middle left (yes/no)?":
            return True if self.built_up_stats["cell_4"].get("built-up class coverage %",False) and (self.built_up_stats["cell_4"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_4"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in center (yes/no)?":
            return True if self.built_up_stats["cell_5"].get("built-up class coverage %",False) and (self.built_up_stats["cell_5"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_5"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in middle right (yes/no)?":
            return True if self.built_up_stats["cell_6"].get("built-up class coverage %",False) and (self.built_up_stats["cell_6"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_6"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in lower left (yes/no)?":
            return True if self.built_up_stats["cell_7"].get("built-up class coverage %",False) and (self.built_up_stats["cell_7"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_7"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in lower middle (yes/no)?":
            return True if self.built_up_stats["cell_8"].get("built-up class coverage %",False) and (self.built_up_stats["cell_8"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_8"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is built-up present in lower right (yes/no)?":
            return True if self.built_up_stats["cell_9"].get("built-up class coverage %",False) and (self.built_up_stats["cell_9"]["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) > 0 or self.built_up_stats["cell_9"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0) else False

        elif ques == "Is dense built-up present in upper left (yes/no)?":
            return True if self.built_up_stats["cell_1"].get("built-up class coverage %",False) and self.built_up_stats["cell_1"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in upper middle (yes/no)?":
            return True if self.built_up_stats["cell_2"].get("built-up class coverage %",False) and self.built_up_stats["cell_2"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in upper right (yes/no)?":
            return True if self.built_up_stats["cell_3"].get("built-up class coverage %",False) and self.built_up_stats["cell_3"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in middle left (yes/no)?":
            return True if self.built_up_stats["cell_4"].get("built-up class coverage %",False) and self.built_up_stats["cell_4"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in center (yes/no)?":
            return True if self.built_up_stats["cell_5"].get("built-up class coverage %",False) and self.built_up_stats["cell_5"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in middle right (yes/no)?":
            return True if self.built_up_stats["cell_6"].get("built-up class coverage %",False) and self.built_up_stats["cell_6"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in lower left (yes/no)?":
            return True if self.built_up_stats["cell_7"].get("built-up class coverage %",False) and self.built_up_stats["cell_7"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in lower middle (yes/no)?":
            return True if self.built_up_stats["cell_8"].get("built-up class coverage %",False) and self.built_up_stats["cell_8"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        elif ques == "Is dense built-up present in lower right (yes/no)?":
            return True if self.built_up_stats["cell_9"].get("built-up class coverage %",False) and self.built_up_stats["cell_9"]["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) > 0 else False

        # elif ques == "Is water present in upper left?":
        #     pass
        # elif ques == "Is water present in upper middle?":
        #     pass
        # elif ques == "Is water present in upper right?":
        #     pass
        # elif ques == "Is water present in middle left?":
        #     pass
        # elif ques == "Is water present in center?":
        #     pass
        # elif ques == "Is water present in middle right?":
        #     pass
        # elif ques == "Is water present in lower left?":
        #     pass
        # elif ques == "Is water present in lower middle?":
        #     pass
        # elif ques == "Is water present in lower right?":
        #     pass

        else: raise ValueError(f"{ques} not found in object presence.")


    def answer_gen_attribute_recognition(
            self,
            ques:str
    ):
        if ques == "Is the majority vegetation healthy or stressed (choose one from 'healthy', 'stressed')?":

            cell_wise_total_veg=list()
            cell_wise_sparse_veg=list()
            cell_wise_dense_veg=list()
            for cell, info in self.veg_stats.items():
                curr_total_veg=0
                for veg_type,coverage in info["vegetation class coverage area in km2"]["coverage_info"].items():
                    if veg_type in ["bare ground or sparse vegetation", "moderate vegetation", "dense vegetation"]:
                        curr_total_veg+=coverage

                    if veg_type == "bare ground or sparse vegetation":
                        cell_wise_sparse_veg.append(coverage)

                    if veg_type == "dense vegetation":
                        cell_wise_dense_veg.append(coverage)

                cell_wise_total_veg.append(curr_total_veg)

            # cell_wise_total_veg = [info["vegetation class coverage area in km2"]["coverage_info"]["bare ground or sparse vegetation"] +
            # info["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) +
            # info["vegetation class coverage area in km2"]["coverage_info"]["dense vegetation"] for info in list(self.veg_stats.values())]

            # cell_wise_sparse_veg = [info["vegetation class coverage area in km2"]["coverage_info"]["bare ground or sparse vegetation"] for info in self.veg_stats.values()]
            #
            # cell_wise_dense_veg = [info["vegetation class coverage area in km2"]["coverage_info"]["dense vegetation"] for info in self.veg_stats.values()]

            total_veg = sum(cell_wise_total_veg)
            total_sparse_veg = sum(cell_wise_sparse_veg)
            total_dense_veg = sum(cell_wise_dense_veg)

            if total_veg==0 or (total_sparse_veg/total_veg) > (total_dense_veg/total_sparse_veg): return "stressed"
            else: return "healthy"


        # elif ques == "Is majority vegetation natural or agricultural?":
        #     pass

        elif ques == "Is the majority built-up low, medium, or high density (choose one from 'low density', 'medium density', 'high density')?":

            cell_no_built_up_area = [(info["built-up class coverage %"]["coverage"]["non built-up"] * info["total area covered by cell in km2"] / 100) for info in list(self.built_up_stats.values()) if info.get("built-up class coverage %", False)]

            cell_medium_built_up_area = [(info["built-up class coverage %"]["coverage"].get("Medium-density built-up/stabilized desert",0) * info["total area covered by cell in km2"] / 100) for info in list(self.built_up_stats.values()) if info.get("built-up class coverage %", False)]

            cell_dense_built_up_area = [(info["built-up class coverage %"]["coverage"].get("High-density built-up areas",0) * info["total area covered by cell in km2"] / 100) for info in list(self.built_up_stats.values()) if info.get("built-up class coverage %", False)]

            total_no_built_up_area = sum(cell_no_built_up_area) if len(cell_no_built_up_area)>0 else 0
            total_medium_built_up_area = sum(cell_medium_built_up_area) if len(cell_medium_built_up_area)>0 else 0
            total_dense_built_up_area = sum(cell_dense_built_up_area) if len(cell_dense_built_up_area)>0 else 0

            if (total_no_built_up_area >= total_medium_built_up_area) and (total_no_built_up_area >= total_dense_built_up_area):
                return "low density"
            elif (total_medium_built_up_area >= total_no_built_up_area) and (total_medium_built_up_area >= total_dense_built_up_area):
                return "medium density"
            else:
                return "high density"


        # elif ques == "Is majority water body fresh, saline, or brackish?":
        #     pass
        #
        # elif ques == "Is majority water body stagnant or flowing?":
        #     pass

        elif ques == "Is the majority  vegetation sparse, moderate, or dense (choose one from 'sparse', 'moderate', 'dense')?":
            # cell_wise_total_veg = [info["vegetation class coverage area in km2"]["coverage_info"]["bare ground or sparse vegetation"] +
            # info["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) +
            # info["vegetation class coverage area in km2"]["coverage_info"]["dense vegetation"] for info in list(self.veg_stats.values())]
            #
            # cell_wise_sparse_veg = [info["vegetation class coverage area in km2"]["coverage_info"]["bare ground or sparse vegetation"] for info in self.veg_stats.values()]
            #
            # cell_wise_moderate_veg = [info["vegetation class coverage area in km2"]["coverage_info"].get("moderate vegetation",0) for info in self.veg_stats.values()]
            #
            # cell_wise_dense_veg = [info["vegetation class coverage area in km2"]["coverage_info"]["dense vegetation"] for info in self.veg_stats.values()]

            cell_wise_total_veg=list()
            cell_wise_sparse_veg=list()
            cell_wise_moderate_veg=list()
            cell_wise_dense_veg=list()
            for cell, info in self.veg_stats.items():
                curr_total_veg=0
                for veg_type,coverage in info["vegetation class coverage area in km2"]["coverage_info"].items():
                    if veg_type in ["bare ground or sparse vegetation", "moderate vegetation", "dense vegetation"]:
                        curr_total_veg+=coverage

                    if veg_type == "bare ground or sparse vegetation":
                        cell_wise_sparse_veg.append(coverage)

                    elif veg_type == "moderate vegetation":
                        cell_wise_moderate_veg.append(coverage)

                    elif veg_type == "dense vegetation":
                        cell_wise_dense_veg.append(coverage)

                cell_wise_total_veg.append(curr_total_veg)

            total_veg = sum(cell_wise_total_veg)
            total_sparse_veg = sum(cell_wise_sparse_veg)
            total_moderate_veg = sum(cell_wise_moderate_veg)
            total_dense_veg = sum(cell_wise_dense_veg)

            if total_veg==0:
                return "sparse"
            elif (total_sparse_veg >= total_moderate_veg) and (total_sparse_veg >= total_dense_veg):
                return "sparse"
            elif (total_moderate_veg >= total_sparse_veg) and (total_moderate_veg >= total_dense_veg):
                return "moderate"
            else:
                return "dense"


        # elif ques == "Is the majority  vegetation fragmented or continuous?":
        #     cell_mornas_I={ cell: info["global morans I"][0] for cell, info in self.veg_stats.items()  if info["global morans I"][1] < 0.005 }
        #
        #     avg_mornas_I=sum(cell_mornas_I.values())/len(cell_mornas_I)
        #
        #     if avg_mornas_I < 0.3:
        #         return "fragmented"
        #     else:
        #         return "continuous"

        # elif ques == "Is the vegetation homogeneous or heterogeneous?":
        #     pass

        # elif ques == "What is the majority urban density (medium, dense)?":
        #     pass

        # elif ques == "What is the land use at this image?":


        elif ques == "What land cover classes are present?":
            return list(self.overall_stats["over all land cover in km2"]["info"].keys())

        elif ques == "What is the land cover dominance (answer in one word)?":
            land_cover_areas=self.overall_stats["over all land cover in km2"]["info"]

            dominating_class=max(
                land_cover_areas,
                key=lambda k: land_cover_areas[k]
            )

            return dominating_class.replace(" coverage area in km2","")

        elif ques == "What land cover is rare (answer in one word)?":
            land_cover_areas=self.overall_stats["over all land cover in km2"]["info"]

            rare_class=min(
                land_cover_areas,
                key=lambda k: land_cover_areas[k]
            )

            return rare_class.replace(" coverage area in km2","")
        
        elif ques == "Which class is more dominating in the image ('water', 'built-up', 'vegetation')?":
            "Which class is more dominating in the image ('water', 'built-up', 'vegetation')?"
            land_cover_areas=self.overall_stats["over all land cover in km2"]["info"]

            valid_land_cover_areas={}
            for cname,val in land_cover_areas.items():
                if "unclassified" not in cname:
                    valid_land_cover_areas[cname]=val

            dominating_class=max(
                valid_land_cover_areas,
                key=lambda k: valid_land_cover_areas[k]
            )
            sub_class_name=dominating_class.replace(" coverage area in km2","")
            # class_name=""
            if 'built' in sub_class_name:
                return "built-up"
            elif 'vegetation' in sub_class_name:
                return "vegetation"
            elif 'water' in sub_class_name:
                return "water"
            else:
                raise ValueError(f"Unexpected subclass name {sub_class_name} found.")
            # return dominating_class.split(" coverage area in km2")[0]
            # return dominating_class.replace(" coverage area in km2","")

        else: raise ValueError(f"{ques} not found in object presence.")

        # elif ques == "Is the area in drought?":
        #     veg_percent=self.overall_stats["over all land cover %"]["info"]["moderate vegetation coverage"] + \
        #         self.overall_stats["over all land cover %"]["info"]["dense vegetation coverage"]
        #
        #     if veg_percent < 15:
        #         return True
        #     else:
        #         return False


    def answer_gen_image_caption(
            self,
            ques:str
    ):
        return self.old_generated_answers[ques]['answer']

    def answer_gen_attribute_reasoning(
            self,
            ques:str
    ):
        return self.old_generated_answers[ques]['answer']

    def get_ans_type(
            self,
    ):

        parents_dir_paths_dict={
            "sentinel":self.sentinel_path,
            "landsat":self.landsat_path,
            "lis3":self.lis3_path,
            "lis4":self.lis4_path
        }

        ques_ans_type_dict=dict()

        for ques_cat, ques_list in self.questions_dict.items():
            for ques in ques_list:
                ques_ans_type_dict[ques]=[]

        for sensor,parent_dir_path in parents_dir_paths_dict.items():

            bands_dir_names = os.listdir(parent_dir_path)

            for bands_dir_name in bands_dir_names:
                bands_dir_path= os.path.join(
                    parent_dir_path,
                    bands_dir_name
                    )



                ques_ans_v2_path=os.path.join(
                    bands_dir_path,
                    "generated_answers_v2.json")


                with open(ques_ans_v2_path,"r", encoding="utf-8") as f:
                    ques_ans_v2=json.load(f)

                for ques_cat, ques_ans_dict in ques_ans_v2.items():


                    for ques, ans in ques_ans_dict.items():

                        type_list=ques_ans_type_dict[ques].copy()
                        type_list.append(str(type(ans)))
                        ques_ans_type_dict[ques]=list(set(type_list))

        return ques_ans_type_dict


    def generate_answers(
            self,
            parents_dir_paths_dict:dict=None,
    ):
        if parents_dir_paths_dict is None:
            parents_dir_paths_dict={
                # "sentinel":self.sentinel_path,
                # "landsat":self.landsat_path,
                # "lis3":self.lis3_path,
                "lis4":self.lis4_path
            }

        for sensor,parent_dir_path in parents_dir_paths_dict.items():

            bands_dir_names = os.listdir(parent_dir_path)

            for bands_dir_name in bands_dir_names:
                bands_dir_path= os.path.join(
                    parent_dir_path,
                    bands_dir_name
                    )

                # dictionary to store generated answers
                generated_ans_dict=dict()

                water_bodies_info_path=os.path.join(
                    bands_dir_path,
                    "water_bodies_info.json",
                )
                veg_stats_path=os.path.join(
                    bands_dir_path,
                    "veg_stats.json",
                )
                built_up_stats_path=os.path.join(
                    bands_dir_path,
                    "built_up_stats.json",
                )
                overall_land_stats_path=os.path.join(
                    bands_dir_path,
                    "overall_land_stats.json",
                )
                location_info_path=os.path.join(
                    bands_dir_path,
                    "location_info.json",
                )
                bounds_path=os.path.join(
                    bands_dir_path,
                    "bounds.json",
                )

                old_generated_answers_path=os.path.join(
                    bands_dir_path,
                    "generated_answers.json",
                )

                with open(water_bodies_info_path,"r", encoding="utf-8") as f:
                    self.water_bodies_info=json.load(f)

                with open(veg_stats_path,"r", encoding="utf-8") as f:
                    self.veg_stats=json.load(f)

                with open(built_up_stats_path,"r", encoding="utf-8") as f:
                    self.built_up_stats=json.load(f)

                with open(overall_land_stats_path,"r", encoding="utf-8") as f:
                    self.overall_stats=json.load(f)

                with open(location_info_path,"r", encoding="utf-8") as f:
                    self.location_info=json.load(f)

                with open(bounds_path,"r", encoding="utf-8") as f:
                    self.bounds_info=json.load(f)

                with open(old_generated_answers_path,"r", encoding="utf-8") as f:
                    self.old_generated_answers=json.load(f)


                for ques_cat, ques_list in self.questions_dict.items():

                    generated_ans_dict[ques_cat]=dict()

                    if ques_cat=="object_counting":

                        for ques in ques_list:
                            if sensor=='lis4' and "built" in ques:
                                continue
                            print(ques)
                            # print(bands_dir_name)

                            answer=self.answer_gen_object_counting(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)
                            
                    elif ques_cat=="object_area":

                        for ques in ques_list:
                            if sensor=='lis4' and "built" in ques:
                                continue
                            print(ques)
                            # print(bands_dir_name)

                            answer=self.answer_gen_object_area(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)

                    elif ques_cat=="object_localization":

                        for ques in ques_list:
                            if sensor=='lis4' and "built" in ques:
                                continue
                            print(ques)
                            # print(bands_dir_name)
                            answer=self.answer_gen_object_localization(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)

                    elif ques_cat=="object_presence":

                        for ques in ques_list:
                            if sensor=='lis4' and "built" in ques:
                                continue
                            print(ques)
                            # print(bands_dir_name)
                            answer=self.answer_gen_object_presence(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)

                    elif ques_cat=="attribute_recognition":

                        for ques in ques_list:
                            if sensor=='lis4' and "built" in ques:
                                continue
                            print(ques)
                            # print(bands_dir_name)

                            answer=self.answer_gen_attribute_recognition(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)

                    elif ques_cat=="image_caption":

                        for ques in ques_list:
                            if sensor=='lis4' and "built" in ques:
                                continue
                            print(ques)
                            # print(bands_dir_name)

                            answer=self.answer_gen_image_caption(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)

                    elif ques_cat=="attribute_reasoning":
                        for ques in ques_list:
                            # if sensor=='lis4' and "built" in ques:
                            #     continue
                            print(ques)
                            # print(bands_dir_name)

                            answer=self.answer_gen_attribute_reasoning(ques)
                            generated_ans_dict[ques_cat][ques]=answer

                            print(answer)



                    generated_ans_path=os.path.join(
                        bands_dir_path,
                        "generated_answers_v2.json",
                    )
                    with open(generated_ans_path,"w") as f:
                        json.dump(generated_ans_dict, f)

                    # ques_ans_type=self.get_ans_type()
                    # generated_ques_ans_type_path=os.path.join(
                    #     bands_dir_path,
                    #     "generated_ques_ans_type.json",
                    # )
                    # with open(generated_ques_ans_type_path,"w") as f:
                    #     json.dump(ques_ans_type, f)



            print("+"*100)








In [6]:
gen_ans=ans_gen()

In [7]:
gen_ans.generate_answers()

How many distinct water bodies are present (give only a number)?
0
How many distinct land cover classes are present (give only a number)?
4
What is the total water body area (give one word answer in km2)?
0.003475
What percentage of total area is water (give one word answer in %)?
0.01
What is the total vegetation area (give one word answer in km2)?
0.199675
What percentage of area is vegetated (give one word answer in %)?
0.43
Where is the most vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
upper right
Where is the least vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
center
Where is the most dense vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lo

# Changing answers in directory of original model outputs

In [12]:
sentinel_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\all\data\data for uploading in runpod\sentinel".replace("\\","/")
landsat_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\all\data\data for uploading in runpod\landsat".replace("\\","/")
lis3_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\all\data\data for uploading in runpod\lis3".replace("\\","/")
lis4_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\all\data\data for uploading in runpod\lis4".replace("\\","/")

parent_dirs_path_dict={
    "sentinel":sentinel_path,
    "landsat":landsat_path,
    "lis3":lis3_path,
    "lis4":lis4_path
}

gen_ans2=ans_gen()
gen_ans2.generate_answers(
    parents_dir_paths_dict=parent_dirs_path_dict)

How many distinct water bodies are present (give only a number)?
24
How many distinct land cover classes are present (give only a number)?
6
What is the total water body area (give one word answer in km2)?
723.32181668
What percentage of total area is water (give one word answer in %)?
14.88
What is the total vegetation area (give one word answer in km2)?
823.6972521499999
What percentage of area is vegetated (give one word answer in %)?
16.94
What is the total built-up area (give one word answer in km2)?
2438.1021645
What percentage of area is built-up (give one word answer in %)?
50.14
Where is the most vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
center
Where is the least vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
low

# Changing answers in directory of fine-tuned model outputs

In [9]:
sentinel_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\remote_sensing_Qwen_answers\complete\data\data for uploading in runpod\sentinel".replace("\\","/")
landsat_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\remote_sensing_Qwen_answers\complete\data\data for uploading in runpod\landsat".replace("\\","/")
lis3_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\remote_sensing_Qwen_answers\complete\data\data for uploading in runpod\lis3".replace("\\","/")
lis4_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\runpod_model_outputs\13_jan_2026\remote_sensing_Qwen_answers\complete\data\data for uploading in runpod\lis4".replace("\\","/")

parent_dirs_path_dict={
    # "sentinel":sentinel_path,
    # "landsat":landsat_path,
    # "lis3":lis3_path,
    "lis4":lis4_path
}

gen_ans2=ans_gen()
gen_ans2.generate_answers(
    parents_dir_paths_dict=parent_dirs_path_dict)

How many distinct water bodies are present (give only a number)?
0
How many distinct land cover classes are present (give only a number)?
4
What is the total water body area (give one word answer in km2)?
0.003475
What percentage of total area is water (give one word answer in %)?
0.01
What is the total vegetation area (give one word answer in km2)?
0.199675
What percentage of area is vegetated (give one word answer in %)?
0.43
Where is the most vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
upper right
Where is the least vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
center
Where is the most dense vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lo

# Changing answer in directory "data for uploading in runpod"

In [10]:
sentinel_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\data for uploading in runpod\sentinel".replace("\\","/")
landsat_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\data for uploading in runpod\landsat".replace("\\","/")
lis3_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\data for uploading in runpod\lis3".replace("\\","/")
lis4_path=r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\data for uploading in runpod\lis4".replace("\\","/")

parent_dirs_path_dict={
    # "sentinel":sentinel_path,
    # "landsat":landsat_path,
    # "lis3":lis3_path,
    "lis4":lis4_path
}

gen_ans2=ans_gen()
gen_ans2.generate_answers(
    parents_dir_paths_dict=parent_dirs_path_dict)

How many distinct water bodies are present (give only a number)?
0
How many distinct land cover classes are present (give only a number)?
4
What is the total water body area (give one word answer in km2)?
0.003475
What percentage of total area is water (give one word answer in %)?
0.01
What is the total vegetation area (give one word answer in km2)?
0.199675
What percentage of area is vegetated (give one word answer in %)?
0.43
Where is the most vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
upper right
Where is the least vegetation cover in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lower left', 'lower middle', 'lower right')?
center
Where is the most dense vegetation in the image (choose one from 'upper left', 'upper middle', 'upper right', 'middle left', 'center' ,'middle right', 'lo